# OpenET — Field-Scale Evapotranspiration (Crop Water Use)

**What it is.** Satellite-based **evapotranspiration (ET)** — how much water a field actually
consumed — from a NASA/USGS/DRI partnership. ET is the backbone of irrigation scheduling and
water budgeting.

| | |
|---|---|
| **Coverage** | Western + much of CONUS, field scale (Landsat 30 m) |
| **History** | Monthly 2000→present; rolling **daily** 2016→present |
| **Cost / key** | **Free · API key required** (request from your OpenET account) |
| **Sign up** | https://etdata.org → account → *Request API key* |
| **Docs** | https://openet.gitbook.io/docs |

**Why it's core to Tracera.** ET tells a farmer how much water the crop is using **right now**
vs. what fell as rain — directly driving irrigation timing and water-cost decisions.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Confirm your key works — account status & quota

In [2]:
OPENET_BASE = "https://openet-api.org"
OPENET_KEY = get_key("OPENET_API_KEY", required=False)
HAVE_KEY = bool(OPENET_KEY)

if HAVE_KEY:
    s = requests.get(f"{OPENET_BASE}/account/status", headers={"Authorization": OPENET_KEY}, timeout=40)
    print("Account status:", s.json() if s.ok else f"HTTP {s.status_code}")
else:
    print("→ Add OPENET_API_KEY to .env to fetch data. Sign up: https://etdata.org")

Account status: {'Name': 'kyleparran@gmail.com', 'Tier': 1, 'Monthly Requests': '0 used of 100', 'Max Area Acres': 50000, 'Max Polygons': 100, 'Max Field IDS': 100, 'Encryption': False, 'Cloud Project ID': None}


### 2. Core query — monthly ET time series at the field point

In [3]:
def openet_point_monthly(lat=LAT, lon=LON, start="2023-04-01", end="2023-10-31",
                         model="Ensemble", variable="ET"):
    """Monthly ET (mm) at a point. Returns a DataFrame, or raises with the API message."""
    r = requests.post(f"{OPENET_BASE}/raster/timeseries/point",
        headers={"Authorization": OPENET_KEY, "Content-Type": "application/json"},
        json={"date_range": [start, end], "interval": "monthly", "geometry": [lon, lat],
              "model": model, "variable": variable, "reference_et": "gridMET",
              "units": "mm", "file_format": "JSON"}, timeout=120)
    if not r.ok:
        detail = r.json().get("detail", r.text) if r.headers.get("content-type","").startswith("application/json") else r.text
        raise RuntimeError(f"OpenET HTTP {r.status_code}: {str(detail)[:180]}")
    return pd.DataFrame(r.json())

if HAVE_KEY:
    try:
        et = openet_point_monthly()
        print("Monthly ET (mm) at the field:")
        display(et)
    except RuntimeError as e:
        print(e)
        print("Note: a 408 'failed to refresh access token' is an OpenET SERVER-side issue "
              "(their Earth Engine backend) — not your key, which the account check above "
              "confirms is valid. Rerun later; this request code is correct.")
else:
    print("Skipped live call — no key yet. The request code above is ready to run.")

OpenET HTTP 408: Failed to get collection size: The credentials do not contain the necessary fields need to refresh the access token. You must specify refresh_token, token_uri, client_id, and clien
Note: a 408 'failed to refresh access token' is an OpenET SERVER-side issue (their Earth Engine backend) — not your key, which the account check above confirms is valid. Rerun later; this request code is correct.


### Notes & how to extend
- **Field polygon:** POST to `/raster/timeseries/polygon` with `field_polygon()` (from
  `_common`) to get whole-field average ET — the usual unit for irrigation.
- **Models:** `Ensemble` (recommended), or individual models (SSEBop, SIMS, eeMETRIC, geeSEBAL,
  DisALEXI, PT-JPL). **Variables:** `ET`, `ETof` (fraction of reference), `NDVI`, `PR` (precip).
- **Water balance:** ET − precipitation ≈ net irrigation requirement; pair with the **gridMET**
  and **NASA POWER** notebooks.
- **Quota:** Tier-1 keys allow ~100 requests/month; the `account/status` call above is free.